In [ ]:
import random
import json
from datasets import load_dataset, Dataset
from typing import List, Dict

# ========================================================================
# 🎬 데이터셋 분석 및 개요 📖
# 데이터셋 이름: lmms-lab/LLaVA-Video-178K
# 한글 제목: LLaVA 비디오 178K (Video Question Answering Dataset)
# 의미 및 설명: 이 데이터셋은 영상 기반의 질의응답(QA)에 특화된 초대형 멀티모달 데이터셋입니다.
#          단순히 이미지를 보는 것을 넘어, '동작'이 포함된 비디오 클립을 보고 대화형으로 질문하고 답하는
#          AI 모델을 훈련하는 데 사용됩니다. 초보자가 AI의 '사고 과정'을 엿보는 흥미로운 데이터입니다!
#          (Conversations: 대화 기록, Video: 비디오 정보, Data_source: 출처 정보)
# ========================================================================

# --- 설정 값 ---
DATASET_ID = "lmms-lab/LLaVA-Video-178K"
# 초보자가 이해하기 쉽도록 'open_ended' (자유 응답) 스플릿을 기본으로 선택하겠습니다!
PRIMARY_SPLIT = 'open_ended'
SAMPLE_COUNT = 5 # 🚨 성능 확인을 위해 상위 5개 샘플만 분석합니다!

print("🌟 안녕하세요! AI 크리에이터를 꿈꾸는 코딩 튜터가 되어드릴게요! 💖")
print("🚀 복잡한 AI 데이터셋을 함께 탐험해 봅시다. 걱정 마세요, 가장 쉽고 재미있는 방법으로 진행할 거예요!")
print("-" * 70)

# --- 데이터 로드 최적화 루틴 ---
print(f"🔄 1. '{DATASET_ID}' 데이터셋을 로드합니다. (스플릿: {PRIMARY_SPLIT})")

try:
    # 1단계: 스트리밍(Streaming)으로 로드를 시도합니다. (가장 빠르고 메모리 효율적)
    full_dataset = load_dataset(DATASET_ID, split=PRIMARY_SPLIT, streaming=True)
    
    # 스트리밍 모드이므로, 반드시 take()와 반복자(iterator) 패턴을 사용해야 합니다!
    print("✅ 스트리밍 모드 감지! 메모리 걱정 없이 순차적으로 데이터를 처리할 수 있어요. 최고!")
    
    # 2단계: 반복자로 데이터를 준비합니다. (IterableDataset 패턴)
    sample_iterator = full_dataset.take(SAMPLE_COUNT)

except Exception as e:
    # 스트리밍 로드가 실패하거나 특정 환경에서 지원하지 않을 경우, 일반(non-streaming) 모드로 전환
    print(f"⚠️ 경고: 스트리밍 로드 실패 ({e.__class__.__name__}). 일반 Dataset 모드로 전환합니다.")
    try:
        # 3단계: 일반 Dataset 모드로 제한된 샘플을 로드합니다.
        full_dataset = load_dataset(DATASET_ID, split=PRIMARY_SPLIT)
        # List로 변환하여 접근 용이하게 합니다.
        full_dataset = full_dataset.select(range(min(SAMPLE_COUNT * 2, len(full_dataset))))
        sample_iterator = full_dataset # 이미 Dataset 객체이므로 직접 사용 가능
    except Exception as e_fallback:
        print(f"😭 오류: 데이터셋 로드에 실패했습니다. 네트워크를 확인하거나 잠시 후 다시 시도해 주세요. ({e_fallback})")
        exit()

# --- AI 데이터 분석 및 탐색 실습 시작 ---

def analyze_conversation(conversation: List[Dict], turn_number: int):
    """
    대화 기록(conversations)을 분석하여, 질문과 답변의 흐름을 친절하게 출력합니다.
    """
    print(f"\n💡 [Turn {turn_number}: 대화 흐름 분석]")
    for i, turn in enumerate(conversation):
        from_speaker = turn['from']
        value = turn['value']
        
        if from_speaker == 'human':
            print(f"  👤 (사용자 질문): {value[:70]}...")
        elif from_speaker == 'llava':
            print(f"  🤖 (AI 답변): {value[:70]}...")
        else:
            print(f"  ❓ (출처/기타): {value[:70]}...")

def analyze_sample(sample, index: int):
    """
    개별 샘플을 받아 핵심 정보(비디오, 소스, 대화)를 분석하고 구조화합니다.
    """
    print("\n" + "=" * 60)
    print(f"✨ 분석 대상 샘플 #{index + 1} 🔍")
    
    # 1. 비디오와 소스 정보 확인 (What is it about?)
    print(f"[🎬 비디오 출처] data_source: {sample.get('data_source', 'N/A')}")
    print(f"[🎥 비디오 정보] video: {sample.get('video', 'N/A')}")
    
    # 2. 대화 분석 (The Core AI Task)
    conversations = sample.get('conversations', [])
    if conversations:
        analyze_conversation(conversations, index + 1)
    else:
        print("❌ 대화 기록(conversations)이 없습니다. 데이터가 불완전할 수 있어요.")

def main_analysis_loop():
    """
    샘플 반복자(Iterator)를 사용하여 데이터를 순회하며 분석을 수행합니다.
    """
    print("\n" + "=" * 70)
    print("💖 🚀 데이터셋 구조 탐험 시작! (최초", SAMPLE_COUNT, "개의 샘플 분석)")
    print("=" * 70)
    
    sample_index = 0
    
    # 🚨 주의! Iterator 패턴을 사용하여 메모리 초과 없이 순차적으로 샘플을 처리합니다.
    try:
        for sample in sample_iterator:
            analyze_sample(sample, sample_index)
            sample_index += 1
            
    except StopIteration:
        print("\n✅ 모든 샘플 분석이 완료되었습니다! 👏")
    except Exception as e:
        print(f"\n❌ 예상치 못한 오류 발생: {e}")

if __name__ == "__main__":
    main_analysis_loop()